<a href="https://colab.research.google.com/github/ddefbcourses/atividade-04-deep-learning-i-Manuelaamorim/blob/main/assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
!pip install mlflow

In [ ]:
import mlflow

mlflow.set_tracking_uri("file:.//mlruns")

mlflow.set_experiment("assignment_cifar10")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [ ]:
mlflow.set_experiment(
    "assignment"
)

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [ ]:
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    # Carrega o dataset
    (X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

    # Flatten: (50000, 32, 32, 3) -> (50000, 3072)
    X_train_full = X_train_full.reshape(X_train_full.shape[0], -1)
    X_test = X_test.reshape(X_test.shape[0], -1)

    # Normalização (escala de 0 a 1)
    X_train_full = X_train_full.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0

    # Separação Treino e Validação
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=seed
    )

    return X_train, X_val, y_train.flatten(), y_val.flatten()

# Execução
X_train, X_val, y_train, y_val = load_data(seed=42)
print("Dados carregados com sucesso!")
print(f"Shape de X_train: {X_train.shape}")
print(f"Shape de X_val: {X_val.shape}")

Formato original: $32 \times 32 \times 3$ (Altura, Largura, Canais RGB).


Features após flatten: $3072$ features ($32 \times 32 \times 3$).

Por que o flatten é necessário? Uma MLP (Multilayer Perceptron) possui uma camada de entrada composta por neurônios unidimensionais. Ela não consegue processar a estrutura espacial 2D/3D diretamente; cada pixel deve ser uma entrada independente.

Importância da normalização: Facilita a convergência do Gradiente Descendente. Sem ela, pixels com valores altos (255) causariam atualizações de pesos muito bruscas, tornando o treinamento instável ou muito lento.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation='relu', hidden_layers=(100,), learning_rate=0.001, seed=42):

    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=30
    )

    model.fit(X_train, y_train)

    return model

print("Treinando a MLP... Por favor, aguarde.")
meu_modelo = train_mlp(X_train, y_train, activation='relu', hidden_layers=(128,), learning_rate=0.001, seed=42)
print("Sucesso! Modelo treinado.")
print(f"Configuração do modelo: {meu_modelo}")

Quantos parâmetros existem na primeira camada?

A quantidade de parâmetros é calculada pela fórmula: $(N_{features} \times N_{neuronios}) + N_{biases}$.No CIFAR-10, temos 3072 features de entrada. Se usarmos uma primeira camada com 128 neurônios, teremos:$(3072 \times 128) + 128 = 393.344$ parâmetros apenas na primeira conexão.

Qual a função da ativação ReLU?

A função ReLU ($f(x) = \max(0, x)$) atua como um filtro que decide se o neurônio deve "disparar" ou não. Ela zera valores negativos e mantém valores positivos. Sua principal função é permitir o treinamento de redes profundas sem que o sinal do erro desapareça (evita o vanishing gradient), sendo muito mais rápida de computar que a Sigmoide ou Tanh.


Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

Porque a MLP é uma rede "totalmente conectada" (Fully Connected). Isso significa que cada único pixel da imagem precisa ter um peso (conexão) para cada neurônio da próxima camada. Como imagens possuem muitos pixels (3072 no caso do CIFAR-10), o número de conexões cresce de forma explosiva, diferente de redes convolucionais que "reutilizam" pesos.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average='macro'),
        "recall": recall_score(y_test, y_pred, average='macro'),
        "f1_score": f1_score(y_test, y_pred, average='macro')
    }

    df_results = pd.DataFrame([metrics])

    return df_results

print("Avaliando o modelo nos dados de validação...")
resultados_df = evaluate(meu_modelo, X_val, y_val)

display(resultados_df)

O que a accuracy representa?

A acurácia é a métrica mais simples: ela representa a porcentagem total de acertos do modelo. Se a acurácia é 0.40, significa que o modelo classificou corretamente 40% das imagens do dataset.

Qual a diferença entre precision e recall?

Precision (Precisão): Foca na qualidade da predição positiva. Responde: "De todas as imagens que o modelo disse ser 'Cachorro', quantas eram realmente 'Cachorro'?". É importante para evitar falsos positivos.

Recall (Revocação): Foca na quantidade de acertos de uma classe. Responde: "De todas as imagens de 'Cachorro' que existiam no dataset, quantas o modelo conseguiu encontrar?". É importante para evitar falsos negativos.

Em quais situações o f1-score é importante?

O F1-Score é a média harmônica entre precisão e recall. Ele é indispensável quando temos um dataset desbalanceado (ex: muito mais fotos de carros do que de gatos) ou quando queremos um equilíbrio real entre precisão e recall, garantindo que o modelo não esteja sacrificando um pelo outro.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
import mlflow
import time

mlflow.set_tracking_uri("file:///content/mlruns")
mlflow.set_experiment("CIFAR10_Tracking")

def run_experiment(run_name, activation, hidden_layers, lr):
    with mlflow.start_run(run_name=run_name):

        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", str(hidden_layers))
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("max_iter", 30)
        mlflow.log_param("batch_size", "auto")

        start_time = time.time()
        model = train_mlp(X_train, y_train, activation, hidden_layers, lr)
        duration = time.time() - start_time

        res_df = evaluate(model, X_val, y_val)


        mlflow.log_metric("accuracy", res_df["accuracy"].values[0])
        mlflow.log_metric("precision", res_df["precision"].values[0])
        mlflow.log_metric("recall", res_df["recall"].values[0])
        mlflow.log_metric("f1_score", res_df["f1_score"].values[0])
        mlflow.log_metric("training_time", duration)

        print(f"Execução '{run_name}' salva no MLflow!")

run_experiment("Teste_Inicial", "relu", (128, 64), 0.001)

print("\n--- Tabela de Experimentos (MLflow UI alternativa) ---")
display(mlflow.search_runs())

1. Qual experimento apresentou melhor desempenho?

    Teste inicial

2. Qual configuração apresentou maior estabilidade?

    Baseado nos parâmetros registrados, a configuração com Learning Rate de 0.001 e ativação ReLU apresentou maior estabilidade. Isso é evidenciado pelo fato de o modelo ter conseguido convergir para uma acurácia próxima de 48% sem divergir

3. Qual o benefício do rastreamento experimental?

    Comparação Objetiva: Você não precisa "adivinhar" qual modelo foi melhor; os números estão lado a lado.

    eprodutibilidade: Qualquer pessoa pode repetir exatamente o mesmo teste usando os mesmos parâmetros registrados.

    Auditoria: Você consegue ver exatamente quando o teste foi feito e quanto recurso computacional ele consumiu.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
config_fixa = {
    'hidden_layers': (128, 64),
    'lr': 0.001
}

ativacoes = ['logistic', 'tanh', 'relu']

print("Iniciando comparação de funções de ativação...")

for act in ativacoes:
    print(f"\n>>> Testando ativação: {act}")
    run_experiment(
        run_name=f"Q5_Ativacao_{act}",
        activation=act,
        hidden_layers=config_fixa['hidden_layers'],
        lr=config_fixa['lr']
    )

print("\nTodos os experimentos da Questão 5 foram registrados!")

df_q5 = mlflow.search_runs()
display(df_q5[df_q5['tags.mlflow.runName'].str.contains("Q5")][['tags.mlflow.runName', 'metrics.accuracy', 'metrics.training_time']])

Qual ativação apresentou melhor convergência?

A Logistic apresentou a melhor convergência final, atingindo 48,68% de acurácia, seguida de perto pela ReLU (47,72%).

Qual ativação apresentou maior estabilidade?

A ReLU e a Logistic mostraram-se estáveis. No entanto, a Tanh teve a menor acurácia (43,38%), sugerindo que ela pode ter tido mais dificuldade em escapar de mínimos locais ou sofreu com saturação de gradientes mais cedo que as outras.

Houve diferenças significativas no treinamento?

Sim, principalmente no tempo. Note que a Tanh foi a mais rápida (114s), enquanto a ReLU foi a mais demorada (256s). Isso indica que a ReLU provavelmente precisou de mais cálculos internos ou não atingiu o critério de parada antecipada (early stopping) tão rápido quanto as outras, explorando mais o espaço de busca. A diferença de acurácia entre a melhor (Logistic) e a pior (Tanh) foi de aproximadamente 5,3%, o que é significativo em aprendizado de máquina.

Por que a ReLU é amplamente utilizada em Deep Learning?

Apesar de a Logistic ter vencido por pouco neste teste de poucas camadas, a ReLU é a preferida em Deep Learning (redes com muitas camadas) porque, evita o Gradiente Desvanecente: Em redes muito profundas, a Logistic faz o erro "sumir", enquanto a ReLU mantém o sinal forte. Além disso, ela é computacionalmente mais barata e tende a deixar alguns neurônios com valor zero, o que ajuda na regularização do modelo.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
arquiteturas = [
    (32,),
    (64,),
    (128, 64),
    (256, 128)
]

print("Iniciando comparação de arquiteturas...")

for arch in arquiteturas:
    print(f"\n>>> Testando arquitetura: {arch}")

    run_experiment(
        run_name=f"Q6_Arch_{arch}",
        activation='relu',
        hidden_layers=arch,
        lr=0.001
    )

print("\nTodos os experimentos da Questão 6 foram registrados!")

df_q6 = mlflow.search_runs()
filtro_q6 = df_q6[df_q6['tags.mlflow.runName'].str.contains("Q6")]
display(filtro_q6[['tags.mlflow.runName', 'metrics.accuracy', 'metrics.training_time']])

Redes maiores sempre melhoraram os resultados?

Neste experimento, sim. Houve um aumento constante na acurácia conforme adicionamos neurônios e camadas. A rede saltou de 36,7% (32 neurônios) para 51,57% (256x128 neurônios). Isso indica que o CIFAR-10 é um dataset complexo que exige uma maior capacidade de memorização e extração de características que apenas redes maiores possuem.

Qual arquitetura apresentou melhor tradeoff (custo-benefício)?

A arquitetura (128, 64) parece ser o melhor equilíbrio. Ela entregou 47,72% de acurácia em 256 segundos. Embora a (256, 128) seja melhor em acurácia, ela levou quase o dobro do tempo (471 segundos) para um ganho de apenas ~4%. Para muitas aplicações, o custo extra de tempo não justificaria esse pequeno ganho.

Houve sinais de overfitting?

Embora a tabela mostre apenas a acurácia de validação, a arquitetura (256, 128) é a principal candidata ao overfitting. Note que ela tem o maior poder computacional. Em MLPs, quando a rede fica muito grande, ela tende a "decorar" os pixels do treino. O fato de ela ter atingido o melhor resultado sugere que ela aprendeu bem, mas precisaríamos comparar com a acurácia de treino para confirmar o overfitting.

Qual arquitetura apresentou maior custo computacional?

Inquestionavelmente a (256, 128), com um tempo de treinamento de 471,05 segundos. Isso ocorre porque o número de parâmetros (pesos) em uma MLP cresce de forma explosiva: cada um dos 3072 pixels de entrada precisa se conectar a todos os 256 neurônios da primeira camada, gerando milhões de operações matemáticas por época.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
lrs = [0.1, 0.01, 0.001]

print("Iniciando comparação de Learning Rates...")

for lr in lrs:
    print(f"\n>>> Testando Learning Rate: {lr}")
    run_experiment(
        run_name=f"Q7_LR_{lr}",
        activation='relu',
        hidden_layers=(128, 64),
        lr=lr
    )

print("\nTodos os experimentos da Questão 7 foram registrados!")

df_q7 = mlflow.search_runs()
filtro_q7 = df_q7[df_q7['tags.mlflow.runName'].str.contains("Q7")]
display(filtro_q7[['tags.mlflow.runName', 'metrics.accuracy', 'params.learning_rate']])

Qual learning rate apresentou melhor desempenho?

O 0.001. Ele é pequeno o suficiente para não fazer o modelo divergir e grande o suficiente para o modelo aprender em um tempo razoável. Ele permite que o modelo encontre os mínimos de erro com precisão.

Qual apresentou maior instabilidade?

O 0.1. Um learning rate de 0.1 é considerado extremamente alto para imagens.

O que acontece quando o learning rate é muito alto?

Ocorre o fenômeno da divergência. Os pesos da rede sofrem atualizações tão drásticas que a função de perda (loss) pode explodir ou o modelo pode nunca aprender nada de útil. Na tabela, você veria uma acurácia muito baixa e um tempo de treino possivelmente curto (porque o modelo "desiste" ou atinge o erro máximo rápido).

O que acontece quando o learning rate é muito baixo? (Ex: 0.0001 ou menor)

O modelo torna-se extremamente lento para convergir. Ele aprenderia de forma muito estável, mas precisaria de centenas de épocas a mais para chegar no mesmo resultado que o LR 0.001 atingiu. Ele corre o risco de ficar preso em um "mínimo local" (um resultado medíocre que ele não tem força para superar).

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

A análise dos experimentos realizados com o dataset CIFAR-10 revela que o treinamento de uma rede perceptron multicamadas (MLP) é um processo sensível ao equilíbrio entre capacidade computacional e ajuste de hiperparâmetros. Durante as execuções, observou-se que o comportamento da função de perda (loss) é fortemente influenciado pelo learning rate. Quando utilizamos uma taxa de 0.001, o aprendizado demonstrou maior estabilidade, permitindo uma convergência gradual e precisa. Em contraste, taxas muito elevadas, como 0.1, frequentemente impedem que o modelo alcance um mínimo de erro, pois os passos na atualização dos pesos são grandes demais, causando instabilidade ou até a divergência do treinamento.

O impacto da arquitetura na performance foi evidente: redes mais profundas e largas, como a configuração (256, 128), apresentaram os melhores resultados de acurácia, atingindo aproximadamente 51,57%. No entanto, esse aumento de performance traz consigo um alto custo computacional, visto que o tempo de treinamento saltou de 38 segundos na rede mais simples para mais de 470 segundos na mais complexa. Esse comportamento do treinamento ilustra o desafio de encontrar um trade-off viável entre o tempo de processamento e o ganho marginal de precisão. Quanto às funções de ativação, embora a Logistic e a ReLU tenham apresentado resultados competitivos neste estudo, a ReLU é amplamente preferida na literatura por sua eficiência em mitigar o problema do desvanecimento do gradiente, permitindo que o sinal do erro flua melhor durante o processo de backpropagation.

A relação entre o backpropagation e o aprendizado é fundamental, pois é através desse mecanismo, baseado na regra da cadeia e no cálculo de gradientes, que a rede consegue ajustar seus pesos de forma iterativa para reduzir o erro na saída. Apesar disso, a MLP demonstrou limitações claras para o processamento de imagens. Por ser uma rede totalmente conectada, ela ignora a hierarquia espacial e a vizinhança entre os pixels, tratando cada entrada de forma isolada. Isso não só gera um número excessivo de parâmetros, aumentando o risco de overfitting, como também impede que o modelo capture padrões visuais complexos (como bordas e texturas) de forma eficiente como as redes convolucionais fariam. Em suma, a configuração de (256, 128) com learning rate de 0.001 obteve o melhor desempenho final, mas as principais dificuldades residiram no tempo de convergência e na sensibilidade do modelo à inicialização dos parâmetros.

1. Qual configuração apresentou melhor resultado final?

    A arquitetura (256, 128), com ativação ReLU (ou Logistic, dependendo da rodada final) e Learning Rate de 0.001. Ela atingiu a maior acurácia de validação (~51,6%).

2. Quais foram as principais dificuldades observadas?

    Custo Computacional: O tempo de treino aumenta drasticamente em redes maiores.

    Convergência Lenta: Atingir acurácias acima de 50% é difícil para uma MLP simples no CIFAR-10.

    Ajuste de Hiperparâmetros: Pequenas mudanças no learning rate podem inutilizar o modelo.

3. Por que MLPs possuem limitações para imagens?


    Imagens possuem correlação espacial (pixels vizinhos formam objetos). As MLPs tratam a imagem como um vetor "achatado" de 3072 números independentes. Isso gera um número excessivo de parâmetros e impede que a rede aprenda padrões como bordas e texturas de forma eficiente, algo que as Redes Convolucionais (CNNs) resolvem.

4. Como o backpropagation contribui para o aprendizado da rede?


    O backpropagation utiliza o cálculo de derivadas (gradientes) e a regra da cadeia para calcular quanto cada peso da rede contribuiu para o erro final. Ele permite que o algoritmo de otimização (como o Adam) saiba exatamente em qual direção e quanto deve ajustar cada um dos milhares de pesos para que, na próxima tentativa, o erro seja menor. Sem ele, a rede não passaria de uma estrutura matemática aleatória e estática.